# Integrated Quantitative Oil Analytics & Bayesian Change Point Detection
This notebook provides the complete analytical baseline, stationarity diagnostics, and Bayesian inference pipeline for analyzing structural breaks in Brent Crude oil prices (2019-Present).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

# 1. Load Brent Oil Daily Spot Prices
df = pd.read_csv('../data/raw/BrentOilPrices.csv', parse_dates=['Date'])
df = df.sort_values('Date').dropna()

# 2. Engineer Daily Log Returns
df['Log_Returns'] = np.log(df['Price'] / df['Price'].shift(1))
df = df.dropna()

# 3. Perform Augmented Dickey-Fuller (ADF) Test to verify Stationarity
result = adfuller(df['Log_Returns'])
print("--- ADF Stationarity Test Results ---")
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4e}")
print(f"Critical Values: {result[4]}")
# If p-value < 0.05, we reject the null hypothesis of non-stationarity

C:\Users\Msara\AppData\Local\Temp\ipykernel_8656\654345862.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv('../data/raw/BrentOilPrices.csv', parse_dates=['Date'])


--- ADF Stationarity Test Results ---
ADF Statistic: -16.4271
p-value: 2.4986e-29
Critical Values: {'1%': np.float64(-3.4310783342658615), '5%': np.float64(-2.861861876398633), '10%': np.float64(-2.566941329781918)}


In [3]:
import pymc as pm

# Setup Bayesian Change Point model using PyMC
with pm.Model() as model:
    # Prior for the discrete change point index (tau)
    tau = pm.DiscreteUniform("tau", lower=0, upper=len(df) - 1)
    
    # Priors for standard deviation before and after the change point
    sigma_1 = pm.HalfNormal("sigma_1", sigma=1)
    sigma_2 = pm.HalfNormal("sigma_2", sigma=1)
    
    # Select sigma based on change point switch
    idx = np.arange(len(df))
    sigma_switch = pm.math.switch(tau >= idx, sigma_1, sigma_2)
    
    # Normal likelihood of daily log returns
    likelihood = pm.Normal("likelihood", mu=0, sigma=sigma_switch, observed=df['Log_Returns'].values)
    
    # Trace sampling via MCMC NUTS
    trace = pm.sample(2000, tune=1000, return_inferencedata=True)

Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>Metropolis: [tau]
>NUTS: [sigma_1, sigma_2]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 22 seconds.
